# NB_hospital_daily_summary
**Target:** `caresyncdatabricks.gold.hospital_daily_summary`  |  **Grain:** one row per hospital per calendar day  |  **Load:** FULL (CREATE OR REPLACE TABLE ... AS SELECT)

**Driving table:** `silver.lab_results` grouped by `hospital_id`, `test_date` (no filters)
* LEFT JOIN `silver.hospitals` ON `hospital_id`
* LEFT JOIN calendar ON `DateKey = date_format(test_date, 'yyyyMMdd')`
* Window: `LAG(total_tests)` partitioned by `hospital_id`, ordered by `summary_date`. This compares against the hospital's previous day that had tests, which is not always the previous calendar day.

Note: the physical calendar table is `silver.calender` and uses PascalCase columns (`DateKey`, `DayName`, `IsWeekend`). They are mapped here to the STTM names `date_key`, `day_name`, `is_weekend`.

In [0]:
%sql
CREATE OR REPLACE TABLE caresyncdatabricks.gold.hospital_daily_summary AS
WITH daily AS (
  SELECT
    hospital_id,
    test_date                            AS summary_date,
    COUNT(*)                             AS total_tests,
    COUNT(DISTINCT patient_id)           AS unique_patients_tested,
    COUNT(DISTINCT doctor_id)            AS active_doctor_count,
    COUNT_IF(result_status = 'Abnormal') AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical') AS critical_test_count
  FROM caresyncdatabricks.silver.lab_results
  GROUP BY hospital_id, test_date
),
with_lag AS (
  SELECT
    *,
    LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) AS prev_day_total_tests
  FROM daily
)
SELECT
  CAST(d.hospital_id AS INT)                                          AS hospital_id,
  h.hospital_name,
  d.summary_date,
  c.DayName                                                           AS day_name,
  c.IsWeekend                                                         AS is_weekend,
  CAST(d.total_tests AS INT)                                          AS total_tests,
  CAST(d.unique_patients_tested AS INT)                               AS unique_patients_tested,
  CAST(d.active_doctor_count AS INT)                                  AS active_doctor_count,
  CAST(d.abnormal_test_count AS INT)                                  AS abnormal_test_count,
  CAST(d.critical_test_count AS INT)                                  AS critical_test_count,
  CAST(d.abnormal_test_count / d.total_tests * 100 AS DECIMAL(5,2))   AS abnormal_rate_pct,
  CAST(d.critical_test_count / d.total_tests * 100 AS DECIMAL(5,2))   AS critical_rate_pct,
  CAST(d.prev_day_total_tests AS INT)                                 AS prev_day_total_tests,
  CAST(
    CASE WHEN d.prev_day_total_tests IS NULL OR d.prev_day_total_tests = 0 THEN NULL
         ELSE (d.total_tests - d.prev_day_total_tests) / d.prev_day_total_tests * 100
    END AS DECIMAL(6,2))                                              AS test_volume_change_pct
FROM with_lag d
LEFT JOIN caresyncdatabricks.silver.hospitals h
  ON d.hospital_id = h.hospital_id
LEFT JOIN caresyncdatabricks.silver.calender c
  ON c.DateKey = CAST(date_format(d.summary_date, 'yyyyMMdd') AS INT)

In [0]:
updated_df = spark.table("caresyncdatabricks.gold.hospital_daily_summary")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))